In [7]:
! python -m pip install gurobipy

In [8]:
from gurobipy import Model, GRB, quicksum, read
import pandas as pd

In [9]:
df = pd.read_csv("dieta.csv")
df = df.set_index("i")

I = df.drop(columns="domanda").columns.to_list()
J = df.drop(index="costo").index.to_list()

C = pd.to_numeric(df.drop(columns="domanda").loc["costo"]).to_dict()
D = pd.to_numeric(df.drop(index="costo")["domanda"]).to_dict()

print(C)
print(D)

A = df.drop(index="costo").drop(columns="domanda")

nomeistanza = df["domanda"]["costo"]

print(nomeistanza)

nomemodello = f"mincostcover_{nomeistanza}"

{'verdura': 4, 'carne': 10, 'frutta': 7}
{'proteine': 20.0, 'ferro': 30.0, 'calcio': 10.0}
nan


In [10]:
m = Model(nomemodello)

#x = {}
#for i in I:
#  x[i] = m.addVar(name=f"qta_{i}",vtype=GRB.CONTINUOUS,lb=0,ub=GRB.INFINITY)
x = m.addVars(I, vtype=GRB.CONTINUOUS,lb=0,ub=GRB.INFINITY, name="qta")

#for j in J:
#  apporto = quicksum( A[i][j] * x[i] for i in I )
#  m.addConstr( apporto >= D[j], name=f"necessita_{j}")
m.addConstrs( (quicksum( A[i][j] * x[i] for i in I ) >= D[j] for j in J) , name = "richiesta")

m.setObjective(quicksum( C[i] * x[i] for i in I), GRB.MINIMIZE)

m.write(f"{nomemodello}.s.lp")

In [11]:
m.setParam("OutputFlag",0)
m.optimize()

if m.Status in (GRB.OPTIMAL, GRB.SUBOPTIMAL):
  print(f"spendi {m.ObjVal} con il piano di copertura seguente")
  m.setParam("OutputFlag",1)
  m.printAttr("x")
  for i in I:
    print(f"ti conviene usare {x[i].x} di {i} (variabile {x[i].varName})")
else:
  print(f"il solver è uscito con codice errore {m.Status}")

spendi 20.0 con il piano di copertura seguente
Set parameter OutputFlag to value 1

    Variable            x 
-------------------------
qta[verdura]            5 
ti conviene usare 5.0 di verdura (variabile qta[verdura])
ti conviene usare 0.0 di carne (variabile qta[carne])
ti conviene usare 0.0 di frutta (variabile qta[frutta])


In [12]:
modello_dafile = read(f"{nomemodello}.s.lp")

modello_dafile.optimize()

modello_dafile.printAttr("x")

Read LP format model from file mincostcover_nan.s.lp
Reading time = 0.00 seconds
: 3 rows, 3 columns, 9 nonzeros
Gurobi Optimizer version 13.0.3 build v13.0.3rc0 (linux64 - "Ubuntu 24.04.5 LTS")

CPU model: Intel(R) Xeon(R) CPU @ 2.20GHz, instruction set [SSE2|AVX|AVX2]
Thread count: 1 physical cores, 2 logical processors, using up to 2 threads

Optimize a model with 3 rows, 3 columns and 9 nonzeros (Min)
Model fingerprint: 0x13812aed
Model has 3 linear objective coefficients
Coefficient statistics:
  Matrix range     [3e+00, 2e+01]
  Objective range  [4e+00, 1e+01]
  Bounds range     [0e+00, 0e+00]
  RHS range        [1e+01, 3e+01]

Presolve time: 0.01s
Presolved: 3 rows, 3 columns, 9 nonzeros

Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    0.0000000e+00   7.500000e+00   0.000000e+00      0s
       1    2.0000000e+01   0.000000e+00   0.000000e+00      0s

Solved in 1 iterations and 0.02 seconds (0.00 work units)
Optimal objective  2.000000000e+01

    Vari